In [1]:
# rsync -av --progress /scratch/franzihe/input/COMBLE u6030275@notchpeak.chpc.utah.edu:/uufs/chpc.utah.edu/common/home/coopers-group1/Franziska/data 

# rsync -av --progress login.nird.sigma2.no:/projects/NS9600K/data/Met-No_obs/  /uio/kant/geo-metos-u1/franzihe/Documents/input/Met-No_obs

In [2]:
import os
import pathlib
import sys
import socket

hostname = socket.gethostname()
print(hostname)


abs_path = str(pathlib.Path(hostname).parent.absolute())
WORKDIR = abs_path[:- (len(abs_path.split('/')[-2] + abs_path.split('/')[-1])+1)]



if "mimi" in hostname:
    DATA_DIR = "/mn/vann/franzihe/"
    FIG_DIR = "/uio/kant/geo-geofag-u1/franzihe/Documents/Figures/ANX_obs_daily/"
if "ivan" in hostname:
    DATA_DIR = "/mn/vann/franzihe/"
    FIG_DIR = "/uio/kant/geo-geofag-u1/franzihe/Documents/Figures/"

INPUT_DATA_DIR = os.path.join(DATA_DIR, 'input')
OUTPUT_DATA_DIR = os.path.join(DATA_DIR, 'output')
UTILS_DIR = os.path.join(WORKDIR, 'utils')

sys.path.append(UTILS_DIR)

# make figure directory to put in reflectivity processed data
try:
    new_dir = pathlib.Path(FIG_DIR)
    new_dir.mkdir(parents=True, exist_ok=True)
except OSError:
    pass

# make output file directory
nc_dir = os.path.join(OUTPUT_DATA_DIR, 'ANX_obs')
try:
    new_dir = pathlib.Path(nc_dir)
    new_dir.mkdir(parents=True, exist_ok=True)
except OSError:
    pass

mimi.uio.no


## Import python packages

* `Python` environment requirements: file requirements_retrieval.txt
* load `python` packages from imports.py
* load `functions` from functions.py



In [3]:
# supress warnings
import warnings
warnings.filterwarnings('ignore') # don't output warnings

# import packages
from imports import(xr, glob, plt, np, DateFormatter, datetime, cm, pd, da, IMProToo,LinearSegmentedColormap, BoundaryNorm, date2num,time)
# from matplotlib.collections import PathCollection
import functions as fct
version = IMProToo.__version__


In [4]:
# reload imports
%load_ext autoreload
%autoreload 2

In [5]:
sites = ["ANX", ]#"ALOMAR"]
years = [#"2021","2022",
         "2023"]

In [6]:
mrr_in_dict, ds_dict_mrr = {}, {}
met_dict_t, met_dict_wd, met_dict_ws, met_dict_p = {}, {}, {}, {}
ds_dict_met_t, ds_dict_met_wd, ds_dict_met_ws, ds_dict_met_p = {}, {}, {}, {}
# ceil_in_dict, ds_dict_ceil = {}, {}
snowflake_class_in_dict, ds_dict_snowflake_class = {}, {}
# graupel_in_dict, ds_dict_graupel = {}, {}
# aggregate_in_dict, ds_dict_aggregate = {}, {}
# small_particle_in_dict, ds_dict_small_particle = {}, {}
# planar_crystal_in_dict, ds_dict_planar_crystal = {}, {}
# columnar_crystal_in_dict, ds_dict_columnar_crystal = {}, {}
# columnar_planar_in_dict, ds_dict_columnar_planar = {}, {}


In [7]:
dicts_to_clean = [ds_dict_mrr,#ds_dict_ceil, 
                  ds_dict_snowflake_class, ds_dict_met_t,ds_dict_met_wd,ds_dict_met_ws,ds_dict_met_p]

In [8]:
met_no_dict = {}
ds_dict_metno = {}



for site in sites:
    for year in years:
        key = f'{site}.{year}'
        
        # Collect file paths
        mrr_in_dict[key]    = sorted(glob(os.path.join(INPUT_DATA_DIR, f'MRR_ANX/maahn/mrr_improtoo*_{site}_{year}*.nc')))
        # ceil_in_dict[key]   = sorted(glob(os.path.join(INPUT_DATA_DIR, f'ceilometer/combined/bsc_cbh_{year}*.nc')))
        snowflake_class_in_dict[key] = sorted(glob(os.path.join(INPUT_DATA_DIR, f'MASC_DB/t_2degC/*{year}_most_common_class_1min_0.6.nc')))

        # hatpro_in_dict[f'{site}.{year}'] = sorted(glob(os.path.join(INPUT_DATA_DIR, f'hatpro/{year}/*/*/*{year}*.nc')))
        
        # Met data handling
        # met_no_dict['{}.{}'.format(site, year)] = sorted(glob(os.path.join(INPUT_DATA_DIR, 'Met-No_obs/SN87110/*{}*'.format(year))))
        if year in ["2021",]:
            met_dict_t[key] = sorted(glob(os.path.join(INPUT_DATA_DIR, f'Met-No_obs/SN87110/air_temperature_{year}*.nc')))
        if year == "2022":
            met_dict_t[key] = sorted(glob(os.path.join(INPUT_DATA_DIR, 'tinytag/TGP-4505_917173.nc')))
        if year == "2023":
            # met_dict_t[key]    = sorted(glob(os.path.join(INPUT_DATA_DIR, f'hatpro/hatpro_new_20240716/level1/{year}/*/*/*.nc')))
            # # met_dict_wd[key]   = sorted(glob(os.path.join(INPUT_DATA_DIR, f'hatpro/hatpro_new_20240716/level1/{year}/*/*/*.nc')))
            # # met_dict_ws[key]   = sorted(glob(os.path.join(INPUT_DATA_DIR, f'hatpro/hatpro_new_20240716/level1/{year}/*/*/*.nc')))
            # # met_dict_t[key] = sorted(glob(os.path.join(INPUT_DATA_DIR, f'Met-No_obs/SN87110/air_temperature_{year}*.nc')))
            # met_dict_wd[key]    = sorted(glob(os.path.join(INPUT_DATA_DIR, f'Met-No_obs/SN87110/ANX_wind_10min_to_1min_filled_{year}.nc')))
            # met_dict_ws[key]    = sorted(glob(os.path.join(INPUT_DATA_DIR, f'Met-No_obs/SN87110/ANX_wind_10min_to_1min_filled_{year}.nc')))
            met_dict_p[key]    = sorted(glob(os.path.join(INPUT_DATA_DIR, f'geonor/precip_*.nc')))
        if year in ['2021', '2022',]:
            met_dict_wd[key] = sorted(glob(os.path.join(INPUT_DATA_DIR, f'Met-No_obs/SN87110/wind_from_direction_{year}*.nc')))
            met_dict_ws[key] = sorted(glob(os.path.join(INPUT_DATA_DIR, f'Met-No_obs/SN87110/wind_speed_{year}*.nc')))
            met_dict_p[key]  = sorted(glob(os.path.join(INPUT_DATA_DIR, f'Met-No_obs/SN87110/*sum(precipitation_amount PT1H)_{year}*')))
        
        # Open or initialize Datasets
        ds_dict_mrr[key]        = fct.open_or_empty(mrr_in_dict[key], decode_times=True, preprocess=lambda x: x.sortby('time'))
        # ds_dict_ceil[key]       = fct.open_or_empty(ceil_in_dict[key], decode_times=True, preprocess=lambda x: x.sortby('time'))
        ds_dict_snowflake_class[key] = fct.open_or_empty(snowflake_class_in_dict[key], decode_times=True, preprocess=lambda x: x.sortby('time'))

        # ds_dict_met_t[key]      = fct.open_or_empty(met_dict_t[key], decode_times=True, preprocess=lambda x: x.sortby('time'))
        # ds_dict_met_wd[key]     = fct.open_or_empty(met_dict_wd[key], decode_times=True, preprocess=lambda x: x.sortby('time'))
        # ds_dict_met_ws[key]     = fct.open_or_empty(met_dict_ws[key], decode_times=True, preprocess=lambda x: x.sortby('time'))
        ds_dict_met_p[key]      = fct.open_or_empty(met_dict_p[key], decode_times=True, preprocess=lambda x: x.sortby('time'))
        
        
        # for ds_dict in dicts_to_clean:
        #     if key in ds_dict: # just in case key is missing
        #         ds = ds_dict[key].copy()
        #         if 'time' in ds.dims and ds.sizes['time'] > 0:
        #             ds_dict[key] = ds.dropna(dim='time')
        # #             ds_dict[key] = ds.sel(time=~ds.indexes['time'].duplicated('first'))
        

In [9]:
# # All snowflake class dicts in a list
# snow_classes = {
#     'graupel': ds_dict_graupel,
#     'aggregate': ds_dict_aggregate,
#     'small_particle': ds_dict_small_particle,
#     'planar_crystal': ds_dict_planar_crystal,
#     'columnar_crystal': ds_dict_columnar_crystal,
#     'columnar_planar_combination': ds_dict_columnar_planar
# }
# markers = ['o', '^', '*', 'h', 's', 'd']
# # Choose a colormap from cmcrameri (e.g., 'batlow', 'lajolla', etc.)
# cmap = cm.lipari

# # Manually define colors from the colormap for each snowflake characteristic
# # Generate 6 evenly spaced values between 0 and 1
# color_positions = np.linspace(0, 1, 6)

# # Map each value to a color in the colormap
# colors = [cmap(pos) for pos in color_positions]
# for year in years:
#     fig, axsm = plt.subplots(6,1,figsize=(10, 5), sharex=True)
#     axs = axsm.flatten()
#     y_level = 1

#     for idx, (class_name, ds_dict) in enumerate(snow_classes.items()):
#         # print(idx, class_name, )#ds_dict)
        
#         key = f"{site}.{year}"  # Construct the key for the current year
#         ds = ds_dict.get(key, None)
        
#         if ds is not None and 'snowflake_class_name' in ds and 'time' in ds:
#             time_values = ds.time.values
#             axs[idx].plot(time_values, [y_level] * len(time_values),
#                         marker=markers[idx],
#                         linestyle='None',
#                         color=colors[idx],
#                         label=class_name,
#                         markersize=4
#                 )
#         else:
#             print(f"No data for {class_name} in {year}")
#         axs[idx].set_yticks([y_level])
#         axs[idx].set_yticklabels([class_name])
#         axs[idx].grid(True)

#     axs[-1].set_xlabel("Datetime")
#     fig.suptitle(f"Snowflake Class Observations in {year}", fontsize=14)
#     fig.tight_layout()#rect=[0, 0.03, 1, 0.95])
#     plt.show()

In [10]:
# for i in range(variable.time.size):
#         idx = da.indexes['time'].get_loc(variable.time.isel(time=i).values,method='nearest')
#         da[idx] = variable.isel(time=i).values
#     return(da)

In [11]:
base_height = np.arange(100,3200, 100) # Predefined MRR profile


In [12]:
# target_dates = {'20210317',}# '20220322', '20230225'}
# target_dates = {'20230215', '20230216','20230217','20230218','20230219','20230220','20230221','20230222','20230223','20230224','20230225','20230226','20230227','20230228','20230301','20230302','20230303','20230305','20230306','20230307','20230308','20230309','20230310','20230311','20230312','20230313','20230314'}
target_dates = {'20230301',}
start = time.time()
for key in list(mrr_in_dict.keys()):
    site, year = key.split('.')
    
    for j, file_path in enumerate(mrr_in_dict[key]):
        date = file_path[-11:-3]
        # if date not in target_dates:
        #     continue  # Skip dates not in the target list
        
        
        
        print(f"Reading file for {site}, {date}...")
        figname = f'mrr_reflectivity_{site}_{date}.png'
        # file_out = os.path.join(OUTPUT_DATA_DIR, f'MRR_ANX/bin_averaged/mrr_improtoo_{version}_{site}_{date}.nc')
        # pathlib.Path(os.path.dirname(file_out)).mkdir(parents=True, exist_ok=True)
        
        # filename = f"obs_{site}_{date}.nc"
        # filename = f"obs_airport_WD_WS_T_{site}_{date}.nc"
        # filename = f"obs_hatpro_airport_WD_WS_T_{site}_{date}.nc"
        filename = f"10min_filled_obs_hatpro_airport_WD_WS_T_{site}_{date}.nc"
        
        # Define start and end time of individual day
        day_start = pd.Timestamp(f"{date} 00:00:00")
        day_end = day_start + pd.Timedelta(hours=23, minutes=59)
        _xmax = day_start + pd.Timedelta(hours=24)
        _time = pd.date_range(start=day_start, periods=1440, freq='min')
        # Create 5-min interval time range for that day
        times = pd.date_range(start=day_start, end=day_end, freq='5min', )#closed='left')

        # create new dataset with NaNs and time resolution 1 minute
        dsx = xr.Dataset(data_vars=dict(
            air_temperature =(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
            wind_direction  =(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
            wind_speed      =(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
            precipitation   =(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
            Ze              =(["time","range"], da.full(shape= (len(_time),len(ds_dict_mrr[key].range)), fill_value = np.nan)),
            W               =(["time","range"], da.full(shape= (len(_time),len(ds_dict_mrr[key].range)), fill_value = np.nan)),
            height          =(["range"], base_height),
            spectralWidth   =(["time","range"], da.full(shape= (len(_time),len(ds_dict_mrr[key].range)), fill_value = np.nan)),
            # bsc             =(["time","altitude"], da.full(shape=(len(_time), len(ds_dict_ceil[key].altitude)), fill_value=np.nan)),
            # cbh             =(["time"], np.full(shape=(len(_time),), fill_value=np.nan)),
            most_common_class = (["time"], np.full(shape=(len(_time),), fill_value=np.nan, dtype=object)),
            aggregate = (["time"], np.full(shape=(len(_time),), fill_value=np.nan,)),
            graupel = (["time"], np.full(shape=(len(_time),), fill_value=np.nan,)),
            pristine = (["time"], np.full(shape=(len(_time),), fill_value=np.nan,)),
            mixed = (["time"], np.full(shape=(len(_time),), fill_value=np.nan,)),
            wind_sector_8 = (["time"], np.full(shape=(len(_time),), fill_value=np.nan, dtype=object)),
            wind_sector_16 = (["time"], np.full(shape=(len(_time),), fill_value=np.nan, dtype=object)),
            riming_deg_level = (["time"], np.full(shape=(len(_time),), fill_value=np.nan,)),
            Matrosov_aggregate = (["time"], np.full(shape=(len(_time),), fill_value=np.nan,)),
            Bracci_aggregate = (["time"], np.full(shape=(len(_time),), fill_value=np.nan,)),
            Nakai_aggregate = (["time"], np.full(shape=(len(_time),), fill_value=np.nan,)),
            Nakai_graupel = (["time"], np.full(shape=(len(_time),), fill_value=np.nan,)),
            Pu_graupel = (["time"], np.full(shape=(len(_time),), fill_value=np.nan,)),
        ),
                         coords=dict(
                             time       =_time,
                             range      =ds_dict_mrr[key].range.values,
                            #  altitude   =ds_dict_ceil[key].altitude.values
                             ))
        

        
# Variable names per year
        if year in ['2021', '2022',]:
            if year == '2021':
                var_id_t    = 'air_temperature'
                var_id_wd   = 'wind_from_direction'
                var_id_pp   = 'sum(precipitation_amount PT1H)'
                var_id_ws   = 'wind_speed'
            elif year == '2022':
                var_id_t    = 'TA'
                var_id_wd   = 'wind_from_direction'
                var_id_pp   = 'sum(precipitation_amount PT1H)'
                var_id_ws   = 'wind_speed'
            # elif year == '2023':
            #     var_id_t    = 'air_temperature'
            #     # var_id_wd   = 'wind_direction'
            #     var_id_wd   = 'wind_direction_filled'
            #     var_id_pp   = 'precipitation'
            #     var_id_ws   = 'wind_speed_filled'
            _air_temp   = ds_dict_met_t[key][var_id_t].sel(time = slice(day_start, day_end))
            _wd         = ds_dict_met_wd[key][var_id_wd].sel(time = slice(day_start, day_end))
            _ws         = ds_dict_met_ws[key][var_id_ws].sel(time = slice(day_start, day_end))
            _pp         = ds_dict_met_p[key][var_id_pp].sel(time = slice(day_start, day_end))
        elif year in ['2023']:
            _air_temp   = ds_dict_snowflake_class[key]['env_T_test'].sel(time = slice(day_start, day_end))
            _wd         = ds_dict_snowflake_class[key]['env_DD_test'].sel(time = slice(day_start, day_end))
            _ws         = ds_dict_snowflake_class[key]['env_FF_test'].sel(time = slice(day_start, day_end))
            _pp         = ds_dict_met_p[key]['precipitation'].sel(time = slice(day_start, day_end))
        
        
        ####
        # # read Ceilometer data
        # _bsc    = ds_dict_ceil[key]['bsc'].sel(time = slice(day_start, day_end))#.rename({"range":"height_c"})
        # _cbh    = ds_dict_ceil[key]['cbh'].sel(time = slice(day_start, day_end))
        
        # read MRR data
        _Ze     = ds_dict_mrr[key]['Ze'].sel(time = slice(day_start, day_end))
                
        _W      = (ds_dict_mrr[key]['W'].sel(time = slice(day_start, day_end)))*(-1)
        _W      = _W.where(_Ze>=-20.)
        
        _sWidth = ds_dict_mrr[key]['spectralWidth'].sel(time = slice(day_start, day_end))
        _sWidth = _sWidth.where(_Ze>=-20.)
        
        _M_agg = ds_dict_mrr[key]['Matrosov_aggregate'].sel(time = slice(day_start, day_end))
        _B_agg = ds_dict_mrr[key]['Bracci_aggregate'].sel(time = slice(day_start, day_end))
        _N_agg = ds_dict_mrr[key]['Nakai_aggregate'].sel(time = slice(day_start, day_end))
        _N_gra = ds_dict_mrr[key]['Nakai_graupel'].sel(time = slice(day_start, day_end))
        _P_gra = ds_dict_mrr[key]['Pu_graupel'].sel(time = slice(day_start, day_end))
        
        # read MASCDB data
        if 'snowflake_class_name' in ds_dict_snowflake_class[key]:
            most_common_class = ds_dict_snowflake_class[key]['snowflake_class_name'].sel(time=slice(day_start, day_end))
            aggregate = ds_dict_snowflake_class[key]['aggregate'].sel(time=slice(day_start, day_end))
            graupel = ds_dict_snowflake_class[key]['graupel'].sel(time=slice(day_start, day_end))
            pristine = ds_dict_snowflake_class[key]['pristine'].sel(time=slice(day_start, day_end))
            wind_dir_sector_8 = ds_dict_snowflake_class[key]['wind_direction_sector'].sel(time=slice(day_start, day_end))
            wind_dir_sector_16 = ds_dict_snowflake_class[key]['compass'].sel(time=slice(day_start, day_end))
            riming_deg_level = ds_dict_snowflake_class[key]['riming_deg_level'].sel(time=slice(day_start, day_end))
        else:
            # Create NaN DataArray
            most_common_class            = fct.empty_class_array('most_common_class', times)
            aggregate = fct.empty_class_array('aggregate', times)
            graupel = fct.empty_class_array('graupel', times)
            pristine = fct.empty_class_array('pristine', times)
            wind_dir_sector_8 = fct.empty_class_array('wind_dir_sector_8', times)
            wind_dir_sector_16 = fct.empty_class_array('wind_dir_sector_16', times)
            riming_deg_level = fct.empty_class_array('riming_deg_level', times)
            print(f"'most_common_class' not found in dataset: {key}")    
            
        if 'mixed' in ds_dict_snowflake_class[key]:
            mixed = ds_dict_snowflake_class[key]['mixed'].sel(time=slice(day_start, day_end)) 
        else: 
            mixed = fct.empty_class_array('mixed', times)
        
        
# minutely averaged ceilometer data
        # _bsc  = _bsc.resample(time='1min').mean('time', skipna=True, keep_attrs=True)
        # _cbh  = _cbh.resample(time='1min').mean('time', skipna=True, keep_attrs=True)
        
        # # Hourly accumulated precipitation
        # if len(_pp) != 0:
        #     _pp    = _pp.resample(time='1H').sum('time', skipna=True, keep_attrs=True)
        
        

        # if year == '2023' and len(_air_temp) != 0:
        #     _air_temp = _air_temp.resample(time='10min').mean('time', skipna=True, keep_attrs=True)
        
        # Fill values to minutely data using function
        dsx['air_temperature']  = fct.arrays_same_time_res(_air_temp, dsx['air_temperature'])
        dsx['wind_direction']   = fct.arrays_same_time_res(_wd, dsx['wind_direction'])
        dsx['wind_speed']       = fct.arrays_same_time_res(_ws, dsx['wind_speed'])
        dsx['precipitation']    = fct.arrays_same_time_res(_pp, dsx['precipitation'])
        dsx['Ze']               = fct.arrays_same_time_res(_Ze, dsx['Ze'])
        dsx['W']                = fct.arrays_same_time_res(_W, dsx['W'])
        dsx['spectralWidth']    = fct.arrays_same_time_res(_sWidth, dsx['spectralWidth'])
        dsx['most_common_class']= fct.arrays_same_time_res(most_common_class, dsx['most_common_class'])
        dsx['aggregate']        = fct.arrays_same_time_res(aggregate, dsx['aggregate'])
        dsx['graupel']          = fct.arrays_same_time_res(graupel, dsx['graupel'])
        dsx['pristine']         = fct.arrays_same_time_res(pristine, dsx['pristine'])
        dsx['mixed']            = fct.arrays_same_time_res(mixed, dsx['mixed'])
        dsx['wind_sector_8']    = fct.arrays_same_time_res(wind_dir_sector_8, dsx['wind_sector_8'])
        dsx['wind_sector_16']   = fct.arrays_same_time_res(wind_dir_sector_16, dsx['wind_sector_16'])
        dsx['riming_deg_level'] = fct.arrays_same_time_res(riming_deg_level, dsx['riming_deg_level'])
        dsx['Matrosov_aggregate'] = fct.arrays_same_time_res(_M_agg, dsx['Matrosov_aggregate'])
        dsx['Bracci_aggregate'] = fct.arrays_same_time_res(_B_agg, dsx['Bracci_aggregate'])
        dsx['Nakai_aggregate']  = fct.arrays_same_time_res(_N_agg, dsx['Nakai_aggregate'])
        dsx['Nakai_graupel']    = fct.arrays_same_time_res(_N_gra, dsx['Nakai_graupel'])
        dsx['Pu_graupel']       = fct.arrays_same_time_res(_P_gra, dsx['Pu_graupel'])
        
        
        
# Assign variable attributes
        for name, var in zip(['air_temperature', 'wind_direction', 'wind_speed', 'precipitation', 'Ze', 'W', 'spectralWidth', 'Matrosov_aggregate', 'Bracci_aggregate', 'Nakai_aggregate', 'Nakai_graupel', 'Pu_graupel'
                            #   'bsc', 'cbh'
                              ], [_air_temp, _wd, _ws, _pp, _Ze, _W, _sWidth, _M_agg, _B_agg, _N_agg, _N_gra, _P_gra #_bsc, _cbh
                                                                                                                                              ]):
            dsx[name].attrs = var.attrs
        
        # test if temperature has same units and assign degC
        units = dsx['air_temperature'].attrs.get('units', '').lower()
        # Check and convert if in Kelvin
        if 'kelvin' in units or 'k' in units:
            print(f"⚠️ Air temperature is in Kelvin. Converting to Celsius... {date}")
            dsx['air_temperature'] = dsx['air_temperature'] - 273.15
            dsx['air_temperature'].attrs['units'] = 'Celsius'
            dsx['air_temperature'].attrs['note'] = 'Converted from Kelvin to Celsius'
        elif 'celsius' in units or 'degc' in units or 'degree_celsius' in units or 'degree_c' in units:
            print(f"✅ Air temperature is already in Celsius. {date}")
        else:
            print(f"❓ Unrecognized units: {units} (no conversion applied). {date}")

        # make units same format
        dsx['W'].attrs['units'] = 'm/s'
        dsx['precipitation'].attrs['units'] = 'mm'
        dsx['air_temperature'].attrs['units'] = '$^{\circ}$C'
        dsx['wind_direction'].attrs['units'] = '$^{\circ}$'


        
        
        # assign attributes to Dataset
        dsx = dsx.assign_attrs({'title': 'Micro rain radar data processed with IMProToo and bin averaged',
                                'comment': ds_dict_mrr[key].attrs['comment'],
                                'institution': ds_dict_mrr[key].attrs['institution'],
                                'contact_person': ds_dict_mrr[key].attrs['contact_person'],
                                'source': ds_dict_mrr[key].attrs['source'],
                                'location': ds_dict_mrr[key].attrs['location'],
                                'history': ds_dict_mrr[key].attrs['history'],
                                'processing_date': ds_dict_mrr[key].attrs['processing_date'],
                                'reference': ds_dict_mrr[key].attrs['reference'],
                                'mrrHeader': ds_dict_mrr[key].attrs['mrrHeader']})
        
        
        # plt_Ze_W_p_T_U(dsx, day_start, _xmax, key)
        
        # # save image to png
        # plt.savefig(FIG_DIR + figname, format='png', bbox_inches='tight', transparent=True,dpi=300)
        
        dsx.to_netcdf(os.path.join(nc_dir, filename))
        # loop over one file
        print("Elapsed time: {:.2f} seconds".format(time.time() - start))

Reading file for ANX, 20230212...
✅ Air temperature is already in Celsius. 20230212
Elapsed time: 2.74 seconds
Reading file for ANX, 20230213...
✅ Air temperature is already in Celsius. 20230213
Elapsed time: 2.90 seconds
Reading file for ANX, 20230214...
✅ Air temperature is already in Celsius. 20230214
Elapsed time: 3.05 seconds
Reading file for ANX, 20230215...
✅ Air temperature is already in Celsius. 20230215
Elapsed time: 3.20 seconds
Reading file for ANX, 20230216...
✅ Air temperature is already in Celsius. 20230216
Elapsed time: 3.34 seconds
Reading file for ANX, 20230217...
✅ Air temperature is already in Celsius. 20230217
Elapsed time: 3.50 seconds
Reading file for ANX, 20230218...
✅ Air temperature is already in Celsius. 20230218
Elapsed time: 3.65 seconds
Reading file for ANX, 20230219...
✅ Air temperature is already in Celsius. 20230219
Elapsed time: 3.81 seconds
Reading file for ANX, 20230220...
✅ Air temperature is already in Celsius. 20230220
Elapsed time: 3.96 seconds
R

In [ ]:
def plot_pcolormesh_with_cbar(ax, fig, data, x, y, cmap, norm, cbar_bounds, cbar_label, cbar_pos):
    pcm = ax.pcolormesh(x, y, data, cmap=cmap, norm=norm, shading='auto')
    cbaxes = fig.add_axes(cbar_pos)
    cbar = plt.colorbar(pcm, cax=cbaxes, shrink=0.5, extend='both', orientation='vertical')
    cbar.ax.set_ylabel(cbar_label)
    if cbar_bounds is not None:
        cbar.set_ticks(cbar_bounds)
    return pcm

def format_range_axis(ax, ds):
    ax.set_yticks(ds['range'][::9])
    ax.set_yticklabels(ds['height'].values[::9])
    ax.set_ylim(0, ds['range'].max())
    ax.set_ylabel("Height [m]")
    
def plot_snowflake_classes(ax, ds):
    particle_classes = [
    ('aggregate', 1),
    ('columnar_crystal', 2),
    ('columnar_planar_combination', 3),
    ('graupel', 4),
    ('planar_crystal', 5),
    ('small_particle', 6)
]
    markers = ['*', '^',  'h','o',  's', 'd']
    
    for y_pos, (name, level) in enumerate(particle_classes, 1):
        data = ds[name].dropna('time')
        ax.plot(date2num(data['time']), 
                np.full(len(data['time']), level),
                marker=markers[y_pos - 1], linestyle='None',
                color='grey', markersize=4, label=name)
    ax.legend(loc='best', fontsize=8)
    ax.set_yticks(range(1, 7))
    ax.set_yticklabels("") #[name for name, _ in particle_classes])
    ax.grid(True)
    
def get_subplot_title(day_start):
    
    if pd.Timestamp("2023-02-23") <= day_start <= pd.Timestamp("2023-03-13"):
        titles = ["CL61 Ceilometer backscatter",'MRR reflectivity', 'MRR mean Doppler Velocity', 'MRR spectral width', 'Precipitation accumulation', '2-m Air temperature', '10-m Wind', 'Snowflake class (MASCDB)']
    elif pd.Timestamp("2023-02-12") <= day_start <= pd.Timestamp("2023-02-22") or day_start == pd.Timestamp("2023-03-14"):
        titles = ['Airport Ceilometer backscatter','MRR reflectivity', 'MRR mean Doppler Velocity', 'MRR spectral width', 'Precipitation accumulation', '2-m Air temperature', '10-m Wind', 'Snowflake class (MASCDB)']
    else:
        titles = ['Airport Ceilometer backscatter','MRR reflectivity', 'MRR mean Doppler Velocity', 'MRR spectral width', 'Precipitation accumulation', '2-m Air temperature', '10-m Wind', 'Snowflake class (MASCDB)']
    
    return titles

In [ ]:
def plt_Ze_W_p_T_U(ds, day_start, xmax, site):
    # Colorbar layout parameters
    cbar_left = 0.92
    cbar_width = 0.015
    cbar_height = 0.0935
    cbar_top = 0.85  # starting point at the top
    cbar_step = 0.1075  # vertical space between cbar axes
        
    

    # titles = ['Airport Ceilometer backscatter','MRR reflectivity', 'MRR mean Doppler Velocity', 'MRR spectral width', 'Precipitation accumulation', '2-m Air temperature', '10-m Wind', 'Snowflake class (MASCDB)']
    titles = get_subplot_title(day_start)
    fig, axsm = plt.subplots(8, 1, figsize=(8.27, 11.69), sharex=True, )
    axs = axsm.flatten()
    
    
    
    time_vals = date2num(ds['time'].values)
    # Ceilometer
    cmap = cm.lipari_r  # define the colormap
    # extract all colors from the colormap
    cmaplist = [cmap(i) for i in range(cmap.N)]
    # force the first color entry to be white
    cmaplist[0] = (255,255,255, 0.9)
    # create the new map
    cmap = LinearSegmentedColormap.from_list(
    'Custom cmap', cmaplist, 10) # cmap.N
    # define the bins and normalize
    # bounds = np.linspace(0, 7e7, 11)
    bounds = np.linspace(0, 200, 9)
    norm = BoundaryNorm(bounds, cmap.N)
    

    # # ceilometer backscatter   
    # bsc = ds['bsc'].dropna(dim=('altitude'), how='all')/1e-6   # Keep rows with at least 1 valid value along the 'altitude' dimension
    # bsc = bsc.transpose('altitude', 'time')
    # cbh = ds['cbh'].isel(layer = 1).where(ds['cbh'].isel(layer = 1) > -1.)
    # plot_pcolormesh_with_cbar(ax=axs[0], fig=fig, data=bsc, x=time_vals, y=ds['altitude'].where(~np.isnan(bsc)), cmap=cmap, norm=norm, cbar_bounds=bounds, cbar_label=f"Backscatter [{ds['bsc'].attrs['units']}]", cbar_pos=[cbar_left, cbar_top - 0 * cbar_step, cbar_width, cbar_height])

    # plot_pcolormesh_with_cbar(ax=axs[0], fig=fig, data=(ds['bsc']).dropna('altitude').transpose('altitude', 'time'), x=time_vals, y=ds['altitude'], cmap=cmap, norm=norm, cbar_bounds=np.arange(0,1.2e8,0.2e8), cbar_label='Backscatter', cbar_pos=[cbar_left, cbar_top - 0 * cbar_step, cbar_width, cbar_height])
    
    # plot cloud base every 5 minutes
    if pd.Timestamp("2023-02-23") <= day_start <= pd.Timestamp("2023-03-13"):
        cbh_data = ds['cbh'].isel(layer=1).resample(time='5min').nearest()
    else:
        cbh_data = ds['cbh'].isel(layer=0).resample(time='5min').nearest()
    # cbh_data = ds['cbh'].isel(layer=0).resample(time='5min').nearest()
    cbh_attrs = ds['cbh'].attrs['units']
    axs[0].plot(date2num(cbh_data['time']), cbh_data.values, ".",linestyle = '-', markersize=2, alpha=0.3, color='grey', label=f"Cloud Base [{cbh_attrs}]")
    # axs[0].scatter(cbh_data['time'].values, cbh_data.values, marker=".", s=1.5, alpha=0.3, color='grey', label="Cloud Base")

    axs[0].set_yticks(np.arange(0, 5001, 1000))
    axs[0].set_ylim(0, 5000)
    axs[0].set_ylabel("Height [m]")
    axs[0].legend()
    
    # Reflectivity'
    # Define your radar-style reflectivity levels (can match the image closely)
    bounds  = np.array([0, 3.5, 7, 10.5, 14, 17.5, 21, 24.5, 28])
    norm    = BoundaryNorm(boundaries=bounds, ncolors=256)  # hawaii_r has 256 levels
    # Use Crameri colormap (hawaii_r)
    cmap    = cm.hawaii_r

    plot_pcolormesh_with_cbar(ax=axs[1], fig=fig, data=ds['Ze'].transpose('range','time'), x=time_vals, y=ds['range'], cmap=cmap, norm=norm, cbar_bounds=bounds, cbar_label=f"Reflectivity [{ds['Ze'].attrs['units']}]", cbar_pos=[cbar_left, cbar_top -1 *cbar_step, cbar_width, cbar_height])

    format_range_axis(axs[1], ds)

    
    # Doppler velocity
    bounds  = np.arange(-6,7,1)
    norm    = BoundaryNorm(boundaries=bounds, ncolors=256)
    cmap    = cm.vik
    
#     dopp    = axs[2].pcolormesh(time_vals, ds['range'], ds['W'].transpose('range', 'time'), cmap=cmap, norm=norm, shading='auto')
    plot_pcolormesh_with_cbar(ax=axs[2], fig=fig, data=ds['W'].transpose('range', 'time'), x=time_vals, y=ds['range'], cmap=cmap, norm=norm, cbar_bounds=np.arange(-6, 8, 2), cbar_label=f"Doppler Velocity [{ds['W'].attrs['units']}]", cbar_pos=[cbar_left, cbar_top -2 * cbar_step, cbar_width, cbar_height])
    
    format_range_axis(axs[2], ds)

    
    # spectral Width
    bounds  = np.arange(0,1.1,0.1)
    norm    = BoundaryNorm(boundaries=bounds, ncolors=256)
    cmap    = cm.batlow_r
    
#     sWidth  = axs[3].pcolormesh(time_vals, ds['range'], ds['spectralWidth'].transpose('range', 'time'), cmap=cmap, norm=norm, shading='auto')
    plot_pcolormesh_with_cbar(ax=axs[3], fig=fig, data=ds['spectralWidth'].transpose('range', 'time'), x=time_vals, y=ds['range'], cmap=cmap, norm=norm, cbar_bounds=np.arange(0,1.2,0.2), cbar_label=f"Spectral Width [{ds['spectralWidth'].attrs['units']}]", cbar_pos=[cbar_left, cbar_top -3 * cbar_step, cbar_width, cbar_height])
    
    format_range_axis(axs[3], ds)

    

    # Precipitation accumulation hourly
    precip = ds['precipitation'].resample(time='1H').sum('time', skipna=True, keep_attrs=True)
    axs[4].bar(precip.time.values, precip.values, width=np.timedelta64(40, 'm'), align='center', color='tab:blue', label='Valid Precipitation', alpha=0.5)
    # Highlight missing data periods with shaded background
    missing_times = precip['time'][np.where(precip==0.)].values
    for t in missing_times:
            axs[4].axvspan(t- np.timedelta64(20, 'm'), t + np.timedelta64(20, 'm'), color='gray', alpha=0.2)
    axs[4].set_ylabel(f"Precipitation [{ds['precipitation'].attrs['units']}]")
    axs[4].set_ylim([0, 4])
    axs[4].legend()
    
    # 2m Temperature
    temp_data = ds['air_temperature'].dropna(dim='time')#.resample(time='5min').nearest()
    axs[5].plot(temp_data['time'].values, temp_data.values, color='tab:red')

    axs[5].hlines(y=2., xmin=day_start, xmax=xmax, color='grey')
    axs[5].set_ylabel(f"2m temperature [{ds['air_temperature'].attrs['units']}]")
    axs[5].set_ylim([-11.5, 12.5])


    
    # 10m wind
    wd_data = ds['wind_direction'].dropna(dim='time').resample(time='5min').nearest()
    axs[6].scatter(wd_data['time'].values, wd_data.values, color='tab:blue', s=2.5)
    axs[6].set_ylabel(f"Wind direction [{ds['wind_direction'].attrs['units']}]", color='tab:blue')
    axs[6].tick_params(axis='y', labelcolor='tab:blue')
    axs[6].set_yticks(range(0, 360+90,90))
    axs[6].set_ylim(0,360)
    
    ax2 = axs[6].twinx()
    ws_data = ds['wind_speed'].dropna(dim='time').resample(time='5min').nearest()
    ax2.plot(ws_data['time'].values, ws_data.values, color='tab:green')
    ax2.set_ylabel(f"Wind speed [{ds['wind_speed'].attrs['units']}]", color='tab:green')
    ax2.tick_params(axis='y', labelcolor='tab:green')
    ax2.set_ylim(0,20)

    

    # snowflake class
    plot_snowflake_classes(axs[7], ds)

    # Shared formatting
    date_form = DateFormatter("%H:00")
    for ax, title in zip(axs, titles):
        ax.set(xlabel = "", title=title)
        ax.xaxis.grid()
        ax.tick_params(axis='x', labelrotation=0)
        plt.setp(ax.get_xticklabels(), horizontalalignment="center")
        ax.xaxis_date()
        ax.xaxis.set_major_formatter(date_form)
        ax.set_xlim([day_start, xmax])

    axs[-1].set(xlabel="Time [UTC]")
    fig.suptitle(f"{site.split('.')[0]} ({ds.time.dt.date[0].values})", fontweight="bold")
    fig.subplots_adjust(top=0.94)



In [ ]:
plt_Ze_W_p_T_U(dsx, day_start, _xmax, key)

In [ ]:

                
                
                


                
                try:
                        _Ze.sel(time=dsx.time[0].values).time.size
                except KeyError:
                        print(KeyError)
                        _Ze = _Ze
                        _height = _height
                        _W = _W
                        _sWidth = _sWidth
                else:
                        if _Ze.sel(time=dsx.time[0].values).time.size >1:
                                _l = _Ze.sel(time=dsx.time[0].values).time.size
                                _Ze = _Ze.isel(time=slice((_l-1), 1441))
                        if _height.sel(time=dsx.time[0].values).time.size >1:
                                _l = _height.sel(time=dsx.time[0].values).time.size
                                _height = _height.isel(time=slice((_l-1), 1441))
                        if _W.sel(time=dsx.time[0].values).time.size >1:
                                _l = _W.sel(time=dsx.time[0].values).time.size
                                _W = _W.isel(time=slice((_l-1), 1441))
                        if _sWidth.sel(time=dsx.time[0].values).time.size >1:
                                _l = _sWidth.sel(time=dsx.time[0].values).time.size
                                _sWidth = _sWidth.isel(time=slice((_l-1), 1441))
                
                


                
                
                # make units same format
                dsx['W'].attrs['units'] = 'm/s'
                dsx['precipitation'].attrs['units'] = 'mm/h'
                dsx['air_temperature'].attrs['units'] = '$^{\circ}$C'
                dsx['wind_direction'].attrs['units'] = '$^{\circ}$'





                # plot 'raw' data and save figure
                # fct.plt_Ze_W_p_T_U(dsx, day_start, _xmax, key)
                # save image to png
                # /uio/kant/geo-geofag-u1/franzihe/Documents/Figures/mrr_reflectivity
                # plt.savefig(FIG_DIR + figname, format='png', bbox_inches='tight', transparent=True,dpi=300)
                #########################################################################################



In [ ]:
_pp.plot()

In [ ]:
_pp2 = _pp.diff(dim="time")
_pp2.plot()

In [ ]:
_pp3 = _pp2.resample(time='1H').sum('time', skipna=True, keep_attrs=True)
_pp3.plot()

In [ ]:
date = mrr_in_dict[key][j][-11:-3]
        print("reading file for {}, {}...".format(site, date))
        
        figname = 'mrr_reflectivity_{}_{}.png'.format(site,date)
        file_out = "{}_{}_{}_{}.nc".format(os.path.join(OUTPUT_DATA_DIR, 'MRR_ANX/bin_averaged/mrr_improtoo'), version, site, date)

        try:
                new_dir = pathlib.Path(os.path.join(OUTPUT_DATA_DIR, 'MRR_ANX/bin_averaged/'))
                new_dir.mkdir(parents=True, exist_ok=True)
        except OSError:
                pass
        
        # Define start and end time of individual day
        day_start = datetime.datetime.strptime("{} 00:00:00".format(date), '%Y%m%d %H:%M:%S').strftime('%Y-%m-%d %H:%M:%S')
        # print(day_start)
        day_end = (datetime.datetime.strptime("{} 00:00:00".format(date), '%Y%m%d %H:%M:%S') + datetime.timedelta(hours=23, minutes=59)).strftime('%Y-%m-%d %H:%M:%S')
        _xmax = (datetime.datetime.strptime(day_end, '%Y-%m-%d %H:%M:%S') + datetime.timedelta(minutes=1,)).strftime('%Y-%m-%d %H:%M:%S')

        # # select Met-No data
        if year == "2021" or year == "2023":
                var_id_t = 'air_temperature'
        elif year == "2022":
                var_id_t = 'TA'
        _air_temp = ds_dict_met_t[key][var_id_t].sel(time = slice(day_start, day_end))
        if year == "2021" or year == "2022":
                var_id_wd = 'wind_from_direction'
                var_id_pp = 'sum(precipitation_amount PT1H)'
        elif year == "2023":
                var_id_wd = 'wind_direction'
        _wd = ds_dict_met_wd[key][var_id_wd].sel(time = slice(day_start, day_end))
        _ws = ds_dict_met_ws[key]['wind_speed'].sel(time = slice(day_start, day_end))
        if year == "2021" or year == "2022":
                var_id_p = 'sum(precipitation_amount PT1H)'
        elif year  == "2023":
                var_id_p = 'Precip_Average_mm'
        _pp = ds_dict_met_p[key][var_id_p].sel(time = slice(day_start, day_end))

        


        # create new dataset with time resolution 1 minute
        _time = pd.date_range(day_start, periods=1440, freq="min").strftime('%Y-%m-%dT%H:%M:%S')
        dsx = xr.Dataset(data_vars=dict(
                                        air_temperature=(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
                                        wind_direction=(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
                                        wind_speed=(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
                                        precipitation=(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
                                        Ze=(["time","range"], da.full(shape= (len(_time),len(ds_dict_mrr[key].range)), fill_value = np.nan)),
                                        W=(["time","range"], da.full(shape= (len(_time),len(ds_dict_mrr[key].range)), fill_value = np.nan)),
                                        height=(["time","range"], da.full(shape= (len(_time),len(ds_dict_mrr[key].range)), fill_value = np.nan)),
                                        spectralWidth=(["time","range"], da.full(shape= (len(_time),len(ds_dict_mrr[key].range)), fill_value = np.nan)),),
                        coords=dict(time=_time, range=ds_dict_mrr[key].range.values))
        # assign variables and attributes to the dataset
        dsx['time'] = pd.to_datetime(dsx.time)
        for i, _t in enumerate(dsx['height'].time):
                dsx['height'][i, :] = dsx['height'].isel(time=i).fillna(np.array([ 100.,  200.,  300.,  400.,  500.,  600.,  700.,  800.,  900.,
                                                                                  1000., 1100., 1200., 1300., 1400., 1500., 1600., 1700., 1800.,
                                                                                  1900., 2000., 2100., 2200., 2300., 2400., 2500., 2600., 2700.,
                                                                                  2800., 2900., 3000., 3100.]))
        dsx['air_temperature'].attrs = _air_temp.attrs
        dsx['wind_direction'].attrs = _wd.attrs
        dsx['wind_speed'].attrs = _ws.attrs
        # dsx['precipitation'].attrs = _pp.attrs

        

        dsx['air_temperature'] = fct.arrays_same_time_res(_air_temp, dsx['air_temperature'])
        dsx['wind_direction']  = fct.arrays_same_time_res(_wd, dsx['wind_direction'])
        dsx['wind_speed']      = fct.arrays_same_time_res(_ws, dsx['wind_speed'])
        # dsx['precipitation']   = fct.arrays_same_time_res(_pp, dsx['precipitation'])
        # dsx['precipitation']   = dsx['precipitation'].resample(time='1H').sum('time', skipna=True, keep_attrs=True)
        # dsx['precipitation']   = dsx['precipitation'].assign_attrs({'units':'mm h-1'})



        # read MRR data
        _Ze = ds_dict_mrr[key]['Ze'].sel(time = slice(day_start, day_end))
        _height = ds_dict_mrr[key]['height'].sel(time = slice(day_start, day_end))
        
        _W = (ds_dict_mrr[key]['W'].sel(time = slice(day_start, day_end)))*(-1)
        _W = _W.where(_Ze>=-20.)
        _W = _W.assign_attrs(ds_dict_mrr[key]['W'].attrs)
        
        _sWidth = ds_dict_mrr[key]['spectralWidth'].sel(time = slice(day_start, day_end))
        _sWidth = _sWidth.where(_Ze>=-20.)
        _sWidth = _sWidth.assign_attrs(ds_dict_mrr[key]['spectralWidth'].attrs)


        
        try:
                _Ze.sel(time=dsx.time[0].values).time.size
        except KeyError:
                print(KeyError)
                _Ze = _Ze
                _height = _height
                _W = _W
                _sWidth = _sWidth
        else:
                if _Ze.sel(time=dsx.time[0].values).time.size >1:
                        _l = _Ze.sel(time=dsx.time[0].values).time.size
                        _Ze = _Ze.isel(time=slice((_l-1), 1441))
                if _height.sel(time=dsx.time[0].values).time.size >1:
                        _l = _height.sel(time=dsx.time[0].values).time.size
                        _height = _height.isel(time=slice((_l-1), 1441))
                if _W.sel(time=dsx.time[0].values).time.size >1:
                        _l = _W.sel(time=dsx.time[0].values).time.size
                        _W = _W.isel(time=slice((_l-1), 1441))
                if _sWidth.sel(time=dsx.time[0].values).time.size >1:
                        _l = _sWidth.sel(time=dsx.time[0].values).time.size
                        _sWidth = _sWidth.isel(time=slice((_l-1), 1441))
        
        
        
        dsx['Ze'].attrs = _Ze.attrs
        dsx['W'].attrs = _W.attrs
        dsx['height'].attrs = _height.attrs
        dsx['spectralWidth'].attrs = _sWidth.attrs

        
        
        dsx['Ze']              = fct.arrays_same_time_res(_Ze, dsx['Ze'])
        dsx['W']               = fct.arrays_same_time_res(_W, dsx['W'])
        dsx['height']          = fct.arrays_same_time_res(_height, dsx['height'])
        dsx['spectralWidth']   = fct.arrays_same_time_res(_sWidth, dsx['spectralWidth'])

        # assign attributes to Dataset
        dsx = dsx.assign_attrs({'title': 'Micro rain radar data processed with IMProToo and bin averaged',
                                'comment': ds_dict_mrr[key].attrs['comment'],
                                'institution': ds_dict_mrr[key].attrs['institution'],
                                'contact_person': ds_dict_mrr[key].attrs['contact_person'],
                                'source': ds_dict_mrr[key].attrs['source'],
                                'location': ds_dict_mrr[key].attrs['location'],
                                'history': ds_dict_mrr[key].attrs['history'],
                                'processing_date': ds_dict_mrr[key].attrs['processing_date'],
                                'reference': ds_dict_mrr[key].attrs['reference'],
                                'mrrHeader': ds_dict_mrr[key].attrs['mrrHeader']})
        
        

        # plot 'raw' data and save figure
        fct.plt_Ze_W_p_T_U(dsx, day_start, _xmax, key)
        # save image to png
        # /uio/kant/geo-geofag-u1/franzihe/Documents/Figures/mrr_reflectivity
        # plt.savefig(FIG_DIR + figname, format='png', bbox_inches='tight', transparent=True)
        #########################################################################################


        
        

In [ ]:
## Bin averaging
        # create new dataset with time resolution 1 minute to
        _time = pd.date_range(day_start, periods=1440, freq="min").strftime('%Y-%m-%dT%H:%M:%S')
        dsm = xr.Dataset(data_vars=dict(
                                        air_temperature=(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
                                        wind_direction=(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
                                        wind_speed=(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
                                        precipitation=(["time"], da.full(shape= (len(_time),), fill_value = np.nan)),
                                        Ze=(["time","range"], da.full(shape= (len(_time),len(dsx.range[1:-1][::2])), fill_value = np.nan)),
                                        W=(["time","range"], da.full(shape= (len(_time),len(dsx.range[1:-1][::2])), fill_value = np.nan)),
                                        height=(["time","range"], da.full(shape= (len(_time),len(dsx.range[1:-1][::2])), fill_value = np.nan)),
                                        spectralWidth=(["time","range"], da.full(shape= (len(_time),len(dsx.range[1:-1][::2])), fill_value = np.nan)),),
                        coords=dict(time=_time, range=dsx.range[1:-1][::2].values))
        # assign variables and attributes to the dataset
        dsm['time'] = pd.to_datetime(dsm.time)

        # averaged values for each bin, return dataset
        dsm = fct.MRR_bin_averaged_dataset(dsx, dsm)

        # add MetNo data to dataset
        dsm['air_temperature'] = dsx['air_temperature']
        dsm['wind_direction'] = dsx['wind_direction']
        dsm['wind_speed'] = dsx['wind_speed']
        # dsm['precipitation'] = dsx['precipitation']


        fct.plt_Ze_W_p_T_U(dsm, day_start, _xmax, key)
        # save image to png
        plt.savefig(FIG_DIR + 'bin_avg_{}'.format(figname), format='png', bbox_inches='tight', transparent=True)

        # rename variables to indicate mean values
        dsm = dsm.rename({'range':'mean_range', 'Ze':'Ze_mean', 'W':'W_mean', 'height':'height_mean', 'spectralWidth':'spectralWidth_mean'})
        ###########################################################

        # merge original dataset with mean dataset
        dsx = dsx.merge(other=dsm, fill_value=np.nan, combine_attrs='no_conflicts')        


        # print("writing...", file_out)
        # dsx.to_netcdf(file_out)


In [ ]:
variable = _pp
da = dsx['precipitation']

In [ ]:
variable.plot()

In [ ]:
da.plot()

In [ ]:
for i in range(da.time.size):
        difference_array = np.absolute(da.time.isel(time=i) - variable.time)
        idx = difference_array.argmin()
        da[i] = variable.isel(time=idx)

In [ ]:
da.plot()
# dsx['precipitation'].plot()

In [ ]:
fct.plt_Ze_W_p_T_U(dsx, day_start, _xmax, key)

In [ ]:
## Create TinyTag Temperature files with same time res as MRR data
# ds = xr.open_mfdataset(sorted(glob(os.path.join(INPUT_DATA_DIR, 'MRR_islas/bin_averaged/*'))))['time']
# ds_T = xr.open_mfdataset(sorted(glob(os.path.join(INPUT_DATA_DIR, 'islas_met_obs/*'))))
# years 
# months=['03']
# days=np.arange(18,22)
# years, months, days


# for day in days:
#     time = '{}-{}-{}'.format(years[0], months[0], day)
#     print(time)
    
#     TA = ds_T['TA'].where(ds.sel(time=time))
    
#     TA2 = TA.interp(time=ds.sel(time=time))
    
#     TA.plot()
#     TA2.plot()
    
#     TA2.to_netcdf('TinyTag_T_ISLAS{}{}{}.nc'.format(years[0], months[0], day))


In [ ]:
dso = xr.open_dataset('/scratch/franzihe/output/MRR_islas/bin_averaged/mrr_improtoo_0.107_ANX_20210321.nc')

Ze = dso['Ze']
Ze_mean = dso['Ze_mean']

In [ ]:
Ze.plot(x='time')

In [ ]:
Ze_mean.plot(x='time')

In [ ]:
# rsync -a --progress franzihe@mimi.uio.no:/uio/kant/geo-metos-u1/franzihe/Documents/Figures/mrr_reflectivity /uufs/chpc.utah.edu/common/home/u6030275/public_html/Figures

In [ ]:
# rsync -av --progress franzihe@mimi.uio.no:/scratch/franzihe/output/MRR_islas /uufs/chpc.utah.edu/common/home/coopers-group1/Franziska/data/


In [ ]:
# rsync -a --progress login.nird.sigma2.no:/projects/NS9600K/sldam/timelapses_islas2022 /home/franzihe/Documents/Figures

In [ ]:
# files = sorted(glob(os.path.join(INPUT_DATA_DIR, 'Met-No_obs/SN87110/*')))
# # metno = xr.open_dataset('/scratch/franzihe/input/Met-No_obs/SN87110/sum(precipitation_amount PT1H)_202203.nc')
# metno = xr.open_mfdataset(files)

# f, ax = plt.subplots(figsize=[3.18, 5.25], ) 
# (metno.sel(time='2022-04-01')['sum(precipitation_amount PT1H)'].cumsum(dim='time') - metno.sel(time='2022-04-01')['sum(precipitation_amount PT1H)'].isel(time=0)).plot.line(ax =ax, x='time', 
#                                                                                             ylim=[0,20], 
#                                                                                             xlim=['2022-04-01T00:00:00.000000000', '2022-04-02T00:00:00.000000000'],
#                                                                                             label='MetNo obs',add_legend=True)

# # plot legend
# ax.legend(loc='upper right')
# ax.set_ylabel('', )#fontsize = 10.0) # Y label
# ax.set_xlabel('', )#fontsize = 10) # X label

# ax.set_xticklabels([])
# ax.set_yticklabels([])
# ax.grid(True)

# plt.savefig('demo.png', transparent=True)